In [ ]:
import numpy as np
import joblib
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

In [ ]:
# Load embeddings
embeddings = np.load("../embeddings/document_embeddings.npy")

# Load clustering model
gmm = joblib.load("../clustering/gmm_model.pkl")

# Load documents
with open("../data/newsgroups_raw.pkl", "rb") as f:
    data = pickle.load(f)

documents = data["documents"]

print("Embeddings shape:", embeddings.shape)
print("Number of documents:", len(documents))
print("Number of clusters:", gmm.n_components)

In [ ]:
cluster_probs = gmm.predict_proba(embeddings)
cluster_labels = gmm.predict(embeddings)

cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()

print("Documents per cluster:")
print(cluster_counts)

In [ ]:
cluster_id = 3  # you can change this

indices = np.where(cluster_labels == cluster_id)[0][:5]

print(f"Sample documents from cluster {cluster_id}\n")

for i in indices:
    print("-----")
    print(documents[i][:400])

Cluster inspection shows that documents grouped together tend to share similar semantic topics.
For example, clusters often contain discussions around specific themes such as computer hardware,
politics, religion, or sports, demonstrating that the clustering captures meaningful structure
in the corpus rather than arbitrary groupings.

In [ ]:
entropy = -np.sum(cluster_probs * np.log(cluster_probs + 1e-9), axis=1)

boundary_indices = np.argsort(entropy)[-5:]

print("High entropy documents (cluster boundary cases):\n")

for i in boundary_indices:
    print("-----")
    print(documents[i][:400])

High-entropy documents represent cases where the model assigns significant probability to multiple clusters.
This indicates semantic overlap between topics. For example, a discussion about gun legislation may belong
to both politics and firearms clusters. These boundary cases demonstrate that fuzzy clustering captures
the nuanced topic structure of the dataset.

In [ ]:
pca = PCA(n_components=2)
reduced = pca.fit_transform(embeddings)

plt.figure(figsize=(10,7))
sns.scatterplot(
    x=reduced[:,0],
    y=reduced[:,1],
    hue=cluster_labels,
    palette="tab20",
    legend=False,
    s=10
)

plt.title("Cluster Visualization (PCA Projection)")
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.show()

The PCA visualization shows how documents are distributed in embedding space.
While clusters are generally separated, overlap exists between semantically related topics.
This confirms that the fuzzy clustering model captures both distinct topic regions and
areas of semantic overlap within the dataset.